# SOLARIS-A1: Exploratory Data Analysis (Stage 5)

> **CRITICAL SCIENTIFIC NOTICE:**
> **Candidate bursts are not verified solar flares.**
> All candidate transient intervals identified in this notebook are strictly exploratory
> (Status: `CANDIDATE — UNVERIFIED`). Nothing produced here is a label, ground truth, or ML target.
> Event comparison status: `NOT PERFORMED — no independent event records supplied`.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from astropy.io import fits

# Ensure project root is on sys.path
repo_root = Path.cwd().resolve()
if (repo_root / "src").is_dir() and str(repo_root / "src") not in sys.path:
    sys.path.insert(0, str(repo_root / "src"))
elif (repo_root.parent / "src").is_dir() and str(repo_root.parent / "src") not in sys.path:
    sys.path.insert(0, str(repo_root.parent / "src"))

from solar_flare.config import load_config
from solar_flare.eda.variability import variability_stats, rolling_summary, count_histogram_data
from solar_flare.eda.bursts import robust_baseline, candidate_intervals
from solar_flare.eda.event_comparison import compare_candidates_to_events
from solar_flare.eda.plots import (
    plot_light_curve,
    plot_count_distribution,
    plot_coverage_timeline,
    plot_burst_gallery,
)
from solar_flare.quality.gti_coverage import load_gti_intervals

In [ ]:
cfg = load_config()
raw_dir = cfg["resolved_paths"]["raw_dir"]
lc_path = raw_dir / "AL1_SOLEXS_20261018_SDD2_L1.lc.gz"
gti_path = raw_dir / "AL1_SOLEXS_20261018_SDD2_L1.gti.gz"

# Strictly read-only FITS access
with fits.open(lc_path, mode="readonly", memmap=True) as hdul:
    times = np.asarray(hdul[1].data["TIME"], dtype=np.float64)
    counts = np.asarray(hdul[1].data["COUNTS"], dtype=np.float64)

gti_data = load_gti_intervals(gti_path)
print(f"Loaded {len(times)} light-curve samples from {lc_path.name}")
print(f"GTI Status: {gti_data.get('status')} ({gti_data.get('n_intervals')} intervals)")

## 1. Variability Statistics
Computed on finite samples only. Non-finite samples (NaNs) are strictly excluded and recorded separately.

In [ ]:
v_stats = variability_stats(counts)
display(pd.DataFrame([v_stats]))

## 2. Robust Running Baseline & Candidate Burst Detection
> **CANDIDATE BURSTS ARE NOT VERIFIED SOLAR FLARES.**
Contiguous excursions above baseline + 5 sigma with duration >= 10 s are tagged `CANDIDATE — UNVERIFIED`.

In [ ]:
eda_cfg = cfg.get("eda", {})
span_s = float(eda_cfg.get("baseline_span_seconds", 600.0))
thresh_sigma = float(eda_cfg.get("burst_threshold_sigma", 5.0))
min_dur_s = float(eda_cfg.get("burst_min_duration_seconds", 10.0))

baseline = robust_baseline(times, counts, span_seconds=span_s)
cands_df = candidate_intervals(
    times, counts, baseline=baseline,
    threshold_sigma=thresh_sigma,
    min_duration_seconds=min_dur_s
)
print(f"Candidate intervals detected: {len(cands_df)} (Status: CANDIDATE — UNVERIFIED)")
display(cands_df.head(10))

## 3. Event Comparison Check
Verifies whether external ground-truth event records are available.

In [ ]:
comp_result = compare_candidates_to_events(cands_df, events_df=None)
print("Status:", comp_result["status"])
print("Detail:", comp_result["detail"])

## 4. Diagnostic Visualizations

In [ ]:
fig_lc = plot_light_curve(times, counts, gti=gti_data, candidates=cands_df, baseline=baseline)
plt.show()

In [ ]:
fig_dist = plot_count_distribution(counts)
plt.show()

In [ ]:
fig_cov = plot_coverage_timeline(gti=gti_data, times=times)
plt.show()

In [ ]:
fig_gal = plot_burst_gallery(times, counts, candidates=cands_df, max_panels=int(eda_cfg.get("gallery_max_panels", 9)))
plt.show()

## Student observations (to be written by the human)
*(To be completed by the student researcher upon inspecting the figures and candidate gallery above.)*

- **Baseline stability:** 
- **High-count peaks and candidate burst structure:** 
- **GTI boundary interactions:** 
- **Next steps for Stage 6 event catalog sourcing:** 